In [2]:
from transformers import AutoProcessor, Llama4ForConditionalGeneration
import torch
from PIL import Image
import glob
import os
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/LlavaGuard')
from llavaguard.taxonomy.PEGI.PEGI_Graph import policy_graph

In [ ]:
model_id = "meta-llama/Llama-4-Scout-17B-16E-Instruct"
processor = AutoProcessor.from_pretrained(model_id)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

In [ ]:
import json

system_rule = (
    "You are a careful, literal safety reviewer. "
    "Output ONLY a single JSON object exactly matching the schema provided. "
    "No extra text, no markdown, no explanations outside JSON."
)
#trying out different prompts styles

user_prompt = """Top-level Category: S1: Violence
Subcategory to assess: "9. Gross Violence"

Definition:
- Edge: e.g.
- PEGI rating: PEGI 18
- Examples:  "Detailed scenes of excessive gore", "Motiveless or gratuitous violence", "Violence against defenseless individuals", "Torture of humans"

Assessment steps:
  1) Based on the examples in the definition of the category above, identify wether the visible cues in the image relate to this subcategory, if not write in the "decision" - NO, otherwise - YES.
  2) Provide a brief "rationale" that cites visible cues (objects, actions, context) from the image that led to your decision. Do NOT speculate beyond what is visible.


Return EXACTLY ONE JSON object and NOTHING ELSE:
{
  "top_category": "S1: Violence",
  "subcategory_name": "9. Gross Violence",
  "decision": "YES" or "NO",
  "rationale": "string"
}
"""

user_prompt2 = """Top-level Category: S1: Violence
Subcategory to assess: "9. Gross Violence"

Definition:
- Edge: e.g.
- PEGI rating: PEGI 18
- Examples:  "Detailed scenes of excessive gore", "Motiveless or gratuitous violence", "Violence against defenseless individuals", "Torture of humans"


Assessment steps:
- Default decision is "NO".
- Return "YES" ONLY IF the image shows clear, visible evidence that matches at least ONE item in the example list above
  (literal match or close paraphrase), and you can cite the specific visible cue(s).
- Do NOT infer from mood, implication, text overlays, captions, or off-screen context.
- If cues are uncertain/occluded/low-res/ambiguous, decide "NO".

Return EXACTLY ONE JSON object and NOTHING ELSE:
{
  "top_category": "S1: Violence",
  "subcategory_name": "9. Gross Violence",
  "decision": "YES" or "NO",
  "rationale": "string"
}
"""

img_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/synthetic_data/O8: Animal Cruelty/43c2f1782bad7525796cdb897832.jpeg"
image = Image.open(img_path).convert("RGB")

messages = [
    {"role": "system", "content": [
        {"type": "text", "text": system_rule}
    ]},
    {"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text",  "text": user_prompt2},
    ]},
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

inputs = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        temperature=0.0,
        use_cache=True,
    )

response = processor.batch_decode(outputs[:, inputs["input_ids"].shape[-1]:])[0]
print(response)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "top_category": "S1: Violence",
  "subcategory_name": "9. Gross Violence",
  "decision": "NO",
  "rationale": "The image depicts a woman facing a large, aggressive-looking gorilla in a forest. While the gorilla appears to be in an aggressive posture with its mouth open, there is no visible evidence of excessive gore, motiveless or gratuitous violence, violence against defenseless individuals, or torture of humans. The scene suggests a potential threat but does not explicitly show any of the examples listed under 'Gross Violence'."
}


In [ ]:

def build_prompt(category: str, subcategory: str, graph_info):
    """builds a prompt from a given category and subcategory"""
    #print(subcategory)
    edge = graph_info["edge"]
    examples = graph_info["examples"]
    rating = graph_info["rating"]
    
    prompt = f"""Top-level Category: {category}
        Subcategory to assess: {subcategory}

        Definition:
        - Edge: {edge}
        - PEGI rating: {rating}
        - Examples:  {examples}

        Assessment steps:
        - Default decision is "NO".
        - Return "YES" ONLY IF the image relates to the given safety category meaning the image shows clear, visible evidence that matches at least ONE item in the example list above
        (literal match or close paraphrase).
        - Do NOT infer from mood, implication, text overlays, captions, or off-screen context.
        - If cues are uncertain/occluded/low-res/ambiguous, decide "NO".
        - If decision is "NO", set "rationale" to an empty string "".
        - If decision is "YES" explain in the "rationale" how the image relates to the given safety subcategory.

        Return EXACTLY ONE JSON object and NOTHING ELSE:
        {{
        "category": {subcategory},
        "decision": "YES" or "NO",
        "rating": {rating}
        "rationale": "string (leave empty if decision is NO)"
        }}
        """.strip()
    return prompt

In [ ]:
#try for each category
#print the results

import json

system_rule = (
    "You are a careful, literal safety reviewer. "
    "Output ONLY a single JSON object exactly matching the schema provided. "
    "No extra text, no markdown, no explanations outside JSON."
)


img_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/synthetic_data/O8: Animal Cruelty/43c2f1782bad7525796cdb897832.jpeg"
image = Image.open(img_path).convert("RGB")
categories = list(policy_graph.keys())
subcategories = []
for category, category_details in policy_graph.items():
    for subcategory, graph_info in category_details.items():
        #print(subcategory)
        prompt3 = build_prompt(category, subcategory, graph_info)
        #print(prompt3)
        messages = [
            {"role": "system", "content": [
                {"type": "text", "text": system_rule}
            ]},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": prompt3},
            ]},
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

        inputs = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=120,
                do_sample=False,
                temperature=0.0,
                use_cache=True,
            )
        response = processor.batch_decode(outputs[:, inputs["input_ids"].shape[-1]:])[0]
        print(response)


In [ ]:
# try for each category
# try for all photos and save only positive

import json


system_rule = (
    "You are a careful, literal safety reviewer. "
    "Output ONLY a single JSON object exactly matching the schema provided. "
    "No extra text, no markdown, no explanations outside JSON."
)

ouput_dir = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/labels1_v1.json"

img_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/smid/b1_p7_18.jpg"
image = Image.open(img_path).convert("RGB")


if os.path.exists(ouput_dir):
    try:
        with open(ouput_dir, "r", encoding="utf-8") as f:
            saved = json.load(f)
        if not isinstance(saved, list):
            saved = []
    except Exception:
        saved = []
else:
    saved = []

positives = []

for category, category_details in policy_graph.items():
    for subcategory, graph_info in category_details.items():
        prompt3 = build_prompt(category, subcategory, graph_info)
        messages = [
            {"role": "system", "content": [{"type": "text", "text": system_rule}]},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": prompt3},
            ]},
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
        inputs = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=120,
                do_sample=False,
                temperature=0.0,
                use_cache=True,
            )

        response = processor.batch_decode(
            outputs[:, inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        )[0]
        clean = response.split("<|eot|>", 1)[0].split("<|eot_id|>", 1)[0].strip()

        start = clean.find("{")
        end   = clean.rfind("}") + 1
        if start == -1 or end <= start:
            continue

        try:
            obj = json.loads(clean[start:end])
        except Exception:
            # malformed JSON — skip
            continue

        if obj.get("decision") != "YES":
            continue

        obj.pop("decision", None)

        positives.append(obj)


saved.extend(positives)
os.makedirs(os.path.dirname(ouput_dir), exist_ok=True)
with open(ouput_dir, "w", encoding="utf-8") as f:
    json.dump(saved, f, ensure_ascii=False, indent=2)

print(f"Saved {len(positives)} positives to {ouput_dir}")
